# Construction du Pipeline de Machine Learning

### Initialisation de Spark et Chargement des Données

In [101]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()

df = spark.read.csv("../data/db_final.csv", header=True, inferSchema=True)
df.show(5)


+-----------+---+------+---------+-------------+---------+--------------+---------------+------+------+----------------+-----------------+---------------+
|CreditScore|Age|Tenure|  Balance|NumOfProducts|HasCrCard|IsActiveMember|EstimatedSalary|Exited|gender|Geography_France|Geography_Germany|Geography_Spain|
+-----------+---+------+---------+-------------+---------+--------------+---------------+------+------+----------------+-----------------+---------------+
|        619| 42|     2|      0.0|            1|        1|             1|      101348.88|     1|   1.0|               1|                0|              0|
|        608| 41|     1| 83807.86|            1|        0|             1|      112542.58|     0|   1.0|               0|                0|              1|
|        502| 42|     8| 159660.8|            3|        1|             0|      113931.57|     1|   1.0|               1|                0|              0|
|        699| 39|     1|      0.0|            2|        0|            

### Équilibrage des Données par Sous-échantillonnage

In [102]:
from pyspark.sql.functions import col

clients_actifs = df.filter(col("Exited") == 0)

clients_inactifs = df.filter(col("Exited") == 1)

print("Clients actifs :", clients_actifs.count())
print("Clients inactifs :", clients_inactifs.count())


Clients actifs : 7963
Clients inactifs : 2037
Clients inactifs : 2037


In [103]:
ratio = clients_inactifs.count() / clients_actifs.count()

clients_actifs_sampled = clients_actifs.sample(withReplacement=False, fraction=ratio, seed=42)

df_balanced = clients_inactifs.union(clients_actifs_sampled)

print("Clients actifs après sous-échantillonnage :", clients_actifs_sampled.count())
print("Clients inactifs :", clients_inactifs.count())


Clients actifs après sous-échantillonnage : 2119
Clients inactifs : 2037
 2119
Clients inactifs : 2037


### Sélection des Features

In [104]:
features = []
for c in df_balanced.columns:
    if c != "Exited":
        features.append(c)

print("Features utilisées :", features)

Features utilisées : ['CreditScore', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary', 'gender', 'Geography_France', 'Geography_Germany', 'Geography_Spain']


### Configuration du Préprocessing (VectorAssembler et StandardScaler)

In [ ]:
from pyspark.ml.feature import VectorAssembler, StandardScaler

assembler = VectorAssembler(inputCols=features, outputCol="features_vector")

scaler = StandardScaler(inputCol="features_vector", outputCol="features_scaled", withMean=True, withStd=True)


### Configuration du Modèle Random Forest

In [106]:
from pyspark.ml.classification import RandomForestClassifier

rf_model = RandomForestClassifier(labelCol="Exited", featuresCol="features_scaled", numTrees=50, seed=42)


### Construction du Pipeline de Machine Learning

In [107]:
from pyspark.ml import Pipeline

pipeline = Pipeline(stages=[assembler, scaler, rf_model])

### Séparation des données : randomSplit([0.8, 0.2], seed=42)

In [108]:
train, test = df_balanced.randomSplit([0.8, 0.2], seed=42)

print("Taille du train :", train.count())
print("Taille du test :", test.count())


Taille du train : 3375
Taille du test : 781
Taille du test : 781


In [109]:
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml import PipelineModel
from pyspark.sql.functions import col

evaluator = BinaryClassificationEvaluator(
    labelCol="Exited", metricName="areaUnderROC"
)


### Configuration de l'Évaluateur et Validation Croisée

In [110]:
from pyspark.ml.tuning import ParamGridBuilder

paramGrid = ParamGridBuilder() \
    .addGrid(rf_model.numTrees, [50, 100, 150]) \
    .addGrid(rf_model.maxDepth, [5, 10, 15]) \
    .addGrid(rf_model.maxBins, [32, 64]) \
    .addGrid(rf_model.minInstancesPerNode, [1, 2, 5]) \
    .addGrid(rf_model.minInfoGain, [0.0, 0.01, 0.05]) \
    .build()

### Configuration de la Grille de Paramètres (ParamGrid)

In [111]:
cv = CrossValidator(estimator=pipeline,estimatorParamMaps=paramGrid,evaluator=evaluator,numFolds=3)
cv_model = cv.fit(train)


### Prédictions sur les Données de Test

In [ ]:
predictions = cv_model.transform(test)
predictions.select("Exited", "prediction", "probability").show(10)

### Évaluation des Performances du Modèle

In [116]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

accuracy_evaluator = MulticlassClassificationEvaluator(labelCol="Exited", predictionCol="prediction", metricName="accuracy")
accuracy = accuracy_evaluator.evaluate(predictions)
print("Accuracy :", accuracy)

precision_evaluator = MulticlassClassificationEvaluator(labelCol="Exited", predictionCol="prediction", metricName="weightedPrecision")
precision = precision_evaluator.evaluate(predictions)
print("Precision :", precision)

recall_evaluator = MulticlassClassificationEvaluator(labelCol="Exited", predictionCol="prediction", metricName="weightedRecall")
recall = recall_evaluator.evaluate(predictions)
print("Recall :", recall)

f1_evaluator = MulticlassClassificationEvaluator(labelCol="Exited", predictionCol="prediction", metricName="f1")
f1 = f1_evaluator.evaluate(predictions)
print("F1-score :", f1)


Accuracy : 0.7772087067861716
Precision : 0.7781699265604074
Precision : 0.7781699265604074
Recall : 0.7772087067861715
Recall : 0.7772087067861715
F1-score : 0.7774395839092976
F1-score : 0.7774395839092976


### Sauvegarde du Modèle Entraîné

In [ ]:
model_path = "../models/best_pipeline_model"
cv_model.bestModel.write().overwrite().save(model_path)